# Marmousi2 Acoustic bv1.2 Forward Modeling Validation

This notebook validates the forward-modeling side of the Marmousi2 acoustic case. It keeps the original example notebooks unchanged and makes the validation parameters explicit before running any command.

## 1. Paths And Imports

The validation reuses the original Marmousi2 data files but writes outputs under `examples/validation/marmousi2_acoustic_bv12/outputs/`.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ADFWI").exists():
    REPO_ROOT = Path("/liufeng1afs/project/04_Inversion/ADFWI-github")

CASE_DIR = REPO_ROOT / "examples" / "acoustic" / "01-model-test" / "01-Marmousi2"
VALIDATION_DIR = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12"
SCRIPT = VALIDATION_DIR / "scripts" / "run_validation.py"
OUTPUT_ROOT = VALIDATION_DIR / "outputs"

sys.path.insert(0, str(VALIDATION_DIR / "scripts"))
from case_definition import build_source_wavelet, build_survey, load_npz

CASE_DIR

## 2. Parameter Definitions

These are the explicit forward-modeling validation parameters. The command runner receives the same values.

In [ ]:
FORWARD_CONFIG = {
    "device": "npu:0",
    "dtype": "float32",
    "model_file": "true_model.npz",
    "f0": 5.0,
    "forward_shot_index": 0,
    "checkpoint_segments": 10,
}
FORWARD_CONFIG

## 3. Model Definition

The forward check uses `true_model.npz` so the modeled shot is generated from the true Marmousi2 model.

In [ ]:
model_npz = load_npz(CASE_DIR / "data" / "model" / FORWARD_CONFIG["model_file"])
model_summary = {
    "nx": int(model_npz["nx"]),
    "nz": int(model_npz["nz"]),
    "dx": float(model_npz["dx"]),
    "dz": float(model_npz["dz"]),
    "nabc": int(model_npz["nabc"]),
    "vp_shape": list(np.asarray(model_npz["vp"]).shape),
    "vp_min": float(np.asarray(model_npz["vp"]).min()),
    "vp_max": float(np.asarray(model_npz["vp"]).max()),
    "rho_min": float(np.asarray(model_npz["rho"]).min()),
    "rho_max": float(np.asarray(model_npz["rho"]).max()),
}
model_summary

## 4. Observation System Definition

The survey is rebuilt from the saved Marmousi2 observation metadata: source locations, receiver locations, time sampling, and source/receiver component types.

In [ ]:
obs_npz = load_npz(CASE_DIR / "data" / "waveform" / "obs_data.npz")
survey = build_survey(obs_npz, f0=FORWARD_CONFIG["f0"])
survey_summary = {
    "shots": survey.source.num,
    "receivers": survey.receiver.num,
    "nt": survey.source.nt,
    "dt": survey.source.dt,
    "src_x_range": [int(np.min(survey.source.get_loc()[:, 0])), int(np.max(survey.source.get_loc()[:, 0]))],
    "rcv_x_range": [int(np.min(survey.receiver.get_loc()[:, 0])), int(np.max(survey.receiver.get_loc()[:, 0]))],
}
survey_summary

## 5. Wavelet Definition

`build_survey(...)` constructs the source wavelet with `ADFWI.utils.wavelet(nt, dt, f0, amp0=1)` and then applies cumulative trapezoid integration, matching the script path.

In [ ]:
source_wavelet = build_source_wavelet(survey.source.nt, survey.source.dt, FORWARD_CONFIG["f0"])
wavelet_summary = {
    "f0": FORWARD_CONFIG["f0"],
    "nt": int(source_wavelet.shape[0]),
    "dt": survey.source.dt,
    "min": float(source_wavelet.min()),
    "max": float(source_wavelet.max()),
    "norm": float(np.linalg.norm(source_wavelet)),
}
wavelet_summary

## 6. Run Forward Modeling Validation

The default cell is a dry run. Set `RUN_FORWARD_MODELING = True` to execute the backend check and single-shot true-model forward run.

In [ ]:
def run_stage(stage: str, *, dry_run: bool = True, overwrite: bool = False):
    command = [
        sys.executable,
        str(SCRIPT),
        stage,
        "--case-dir", str(CASE_DIR),
        "--device", FORWARD_CONFIG["device"],
        "--dtype", FORWARD_CONFIG["dtype"],
        "--f0", str(FORWARD_CONFIG["f0"]),
        "--forward-model-file", FORWARD_CONFIG["model_file"],
        "--forward-shot-index", str(FORWARD_CONFIG["forward_shot_index"]),
        "--checkpoint-segments", str(FORWARD_CONFIG["checkpoint_segments"]),
        "--output-root", str(OUTPUT_ROOT),
    ]
    if dry_run:
        command.append("--dry-run")
    if overwrite:
        command.append("--overwrite")
    proc = subprocess.run(command, cwd=str(REPO_ROOT), text=True, capture_output=True, check=False)
    if proc.stderr:
        print(proc.stderr)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f"stage {stage} failed with return code {proc.returncode}")
    return json.loads(proc.stdout[proc.stdout.find("{"):])

run_stage("forward", dry_run=True)

In [ ]:
RUN_FORWARD_MODELING = False

if RUN_FORWARD_MODELING:
    check_result = run_stage("check", dry_run=False, overwrite=True)
    forward_result = run_stage("forward", dry_run=False, overwrite=True)
else:
    check_result = {"status": "skipped", "reason": "set RUN_FORWARD_MODELING=True"}
    forward_result = {"status": "skipped", "reason": "set RUN_FORWARD_MODELING=True"}

check_result, forward_result